# 04 — Reverse transfer

Notebook 02 evaluates models developed in MCS when applied to YRBS. That result alone cannot
show whether the observed transfer difficulty is a general difference between the cohorts or
depends on which cohort supplies the training data. I therefore reverse the source and
evaluation cohorts, keeping the rest of the comparison symmetric.

**The question.** When the source and target cohorts are reversed, does unadapted transfer show
a similar loss in performance across model families?

**What is reversed, and only that.** The training cohort and the evaluation cohort. The splits,
the preprocessing, the outcome, the nine families, the twenty seeds, the three thresholds and
every metric definition are the main analysis's own, called from the same library functions.

**The configuration follows the training cohort, here as in the forward direction.** The
YRBS-trained model takes the YRBS consensus selection; the MCS-trained local reference takes the
MCS one. Both were chosen by the same procedure — three development seeds, five-fold inner
cross-validation, AUC — inside each cohort's own outer-training partitions, and both are fixed
across the twenty splits. So the mirror is exact: a fixed YRBS-developed model evaluated twenty
times in MCS, against a fixed MCS-developed model evaluated twenty times in MCS.

**What is deliberately not repeated.** No adaptation of any kind runs here — no fine-tuning,
recalibration, quantile mapping, importance weighting, pseudo-labelling, ensembling or
self-training, and no label-budget, leave-one-pillar-out, subgroup or conformal work. Running
the adaptation battery backwards would answer a different and much larger question, and this
section exists to constrain the explanations for one forward result rather than to duplicate
it.

**Why notebook 02's own results are reused.** The MCS local reference here is the same model, on
the same split, under the same configuration, as notebook 02's source reference. It is read
rather than recomputed. **What is loaded is a set of metric rows and a configuration, never a
fitted model** — the role that crosses the border is fitted here, on its own training partition.

This answers reviewer weakness W1 in `docs/REVIEWER_RESPONSE.md` in part. It cannot separate
country, population and measurement, and it is not offered as doing so.

In [ ]:
import os
import sys
import time
from types import SimpleNamespace

# src/ is a sibling of notebooks/; only one of these two exists from any
# working directory, and a path that does not exist is ignored.
sys.path[:0] = ["src", "../src"]

import numpy as np
import pandas as pd

import config
import data
import evaluation
import inputs
import models
import regime_names
import transfer

# This notebook's results are MCS-derived throughout, so it is never saved with its outputs.
# `scripts/check_notebook_outputs.py` and the pre-commit hook enforce that.
PUBLIC_NOTEBOOK = False

---
## A — Question and design

### The correspondence with the forward analysis

Two roles in each direction, and each pair is the same kind of thing. Read a row across: the
names differ because the cohorts differ, and the quantity does not.

| | forward (notebook 02) | reverse (here) | what it is |
|---|---|---|---|
| transferred model | MCS-developed, tested on YRBS (`unadapted`) | **YRBS-developed, tested on MCS** (`reverse_transfer`) | the model that crosses the border |
| local reference | YRBS-developed, tested on YRBS (`yrbs_local`) | **MCS-developed, tested on MCS** (`mcs_local_reference`) | what the evaluating cohort reaches developing its own model |

**Neither local reference is a ceiling.** Each is a model fitted somewhere, and a transferred
model exceeding one is a result rather than a contradiction.

### The design is symmetric

Both cohorts' configurations come from the same consensus procedure — development seeds 0, 1 and
2; five-fold stratified inner cross-validation within each development seed's outer-training
partition; AUC; the mean of the three seed-level mean AUCs — run separately inside each cohort.
Each produced one fixed configuration per family and threshold, and both are held fixed across
all twenty evaluation splits.

So the forward transfer loss and the reverse transfer loss are the same construction with the
cohorts exchanged, and comparing them carries no caveat about one side having had a different
kind of development budget from the other.

**What both local references do inherit** is notebook 02's Section N limitation: development and
evaluation resamples overlap within each cohort, so respondents held out in one evaluation split
may appear in development partitions under other seeds. That applies to `mcs_local_reference`
here exactly as it applies to `mcs_internal` and `yrbs_local` forward, and it is one reason the
same procedure aligns the two references without guaranteeing that any residual optimism is of
the same magnitude in both.

### One thing the two directions do not share, before any number

The label shift is asymmetric by construction: the YRBS/MCS prevalence ratio grows with the
threshold. A directional difference is therefore not automatically a transferability
difference, and this section reports what it measures rather than what would follow if the two
directions were exchangeable.

In [ ]:
# The scope. Everything below reads these and nothing else sets them.
FAMILIES = regime_names.FAMILIES        # the canonical nine, in reporting order
THRESHOLDS = regime_names.THRESHOLDS    # the three ACE-count cuts
SEEDS = range(20)                       # the twenty-seed protocol; seed 0..19
ROLES = regime_names.REVERSE_ROLES      # reverse_transfer, and the MCS local reference

HEADLINE_THRESHOLD = ">=2"              # the threshold this section narrates
SENSITIVITY_THRESHOLDS = (">=1", ">=3")

_cells = len(FAMILIES) * len(THRESHOLDS) * len(SEEDS)
print(f"{len(FAMILIES)} families x {len(THRESHOLDS)} thresholds x {len(SEEDS)} seeds "
      f"= {_cells:,} cells")
print(f"{len(ROLES)} roles per cell: {', '.join(ROLES)}")
# One fit per cell, not two: the MCS local reference is notebook 02's own source reference and
# is read rather than refitted, and the cross-configured arm the design once carried is retired
# — a model uses the settings selected in the cohort it is trained on.
print(f"{_cells:,} model fits, every one of them on a YRBS training partition")
print(f"{_cells:,} local-reference rows reused from notebook 02, none of them refitted")

---
## B — Inputs and the fixed-setting contract

Four artefacts from notebook 01 — the harmonised features and the five ACE pillars, per cohort
— one tracked specification, and one frame of finished metric rows. Nothing here re-reads a raw
cohort, nothing here selects a configuration, and nothing here recomputes a quantity the
forward battery already carries.

**The splits are the canonical ones.** `data.build_splits` is called with the same arguments
notebook 02 calls it with, so this section inherits the same respondent partitions, the same
ordering, the same missing-outcome handling, the same feature ordering and the same
cohort-specific imputation and standardisation. There is no second split definition in this
notebook.

**Where each role's numbers come from.**

| role | trained on | configuration | source | fitted here |
|---|---|---|---|---|
| `reverse_transfer` | YRBS training side | the YRBS consensus selection | `spec/local_model_settings.csv` | yes |
| `mcs_local_reference` | MCS training side | the MCS consensus selection | notebook 02's own `mcs_internal` rows | **no — reused** |

**A model uses the settings selected in the cohort it is trained on.** `reverse_transfer` is
trained on YRBS, so it takes `models.yrbs_settings()`. Handing it the MCS mapping would make it a
model configured by one cohort's development budget and trained on another's, under a name that
says only the training cohort changed; `transfer.reverse_transfer_scores` refuses to run
unconfigured rather than falling back.

**The second role is the battery's source reference under another name.** Both are a model
fitted on the MCS training side under the MCS consensus selection, scored on the same held-out
MCS records, through the same metric functions. Refitting it would compute a number notebook 02
already computed, on the same rows; the only thing a second computation could establish is that
the two agree. So the rows are read and **validated** — regime, arm, lineage, the MCS-evaluated
flag, the MCS-selected configuration label, and one row per declared cell with every seed
present.

**No fallback anywhere.** The fixed specification is complete by construction, so there is no
unconfigured cell; a battery frame that fails any check stops the read with a message naming what
was wrong. Nothing here substitutes a default, borrows a configuration from another cohort,
quietly recomputes a reused quantity, or reports a mean over whichever seeds survived.

In [ ]:
# THE READ GATE IS ACKNOWLEDGED BEFORE THE KERNEL STARTS, NOT HERE. This notebook reads MCS
# artefacts and sets nothing; the check is a clearer refusal than a resolver's, and it names no
# location.
if os.environ.get("MCS_READ_OK") != "1":
    raise RuntimeError(
        "This notebook reads restricted MCS artefacts. Acknowledge the read gate for the "
        "kernel before running it. Nothing in this notebook sets it.")

# Notebook 01 writes the harmonised feature frames. The guarded reader refuses
# frames created under an older preprocessing version or a different schema.
#
# Only the model schema is held. Every section below fits or scores, and `data.model_features`
# is the gate `data.build_splits` refuses a frame without; the harmonised construct columns are
# read by notebook 01's cohort-association section rather than here.
COHORT = SimpleNamespace(
    X_mcs_model=data.model_features(
        data.read_harmonised_features(config.MCS_FEATURES,
                                      columns=list(data.FEATURE_COLUMNS))),
    X_yrbs_model=data.model_features(
        data.read_harmonised_features(config.YRBS_FEATURES,
                                      columns=list(data.FEATURE_COLUMNS))),
    pillars_mcs=pd.read_parquet(config.MCS_PILLARS),
    pillars_yrbs=pd.read_parquet(config.YRBS_PILLARS),
)

# Composed here rather than stored, so `make_outcome` stays the outcome's one definition.
y_mcs = {t: data.make_outcome(COHORT.pillars_mcs, t) for t in THRESHOLDS}
y_yrbs = {t: data.make_outcome(COHORT.pillars_yrbs, t) for t in THRESHOLDS}

print("read the four cohort artefacts and composed the outcome at each threshold")
print(f"model schema: {len(data.FEATURE_COLUMNS)} harmonised predictors -> "
      f"{len(data.MODEL_FEATURE_COLUMNS)} model columns ({data.PREPROCESSING_VERSION})")

In [ ]:
# The canonical splits, built exactly as notebook 02 builds them. Held for the notebook's life.
splits = {(seed, t): data.build_splits(seed, y_mcs[t], y_yrbs[t],
                                       X_mcs=COHORT.X_mcs_model, X_yrbs=COHORT.X_yrbs_model,
                                       threshold=t)
          for t in THRESHOLDS for seed in SEEDS}


def check_reverse_frames(bundle, seed, threshold):
    """The alignment this section depends on, checked per bundle. Returns a list of problems.

    `build_splits` already refuses an unaligned cohort. What it does not check is the pairing
    this section relies on: the YRBS training side it fits on and the MCS test side it scores,
    each with its own outcome, and both carrying the declared features in the declared order.
    """
    found = []
    where = f"seed {seed}, >={threshold}"
    columns = list(data.MODEL_FEATURE_COLUMNS)
    for name in ("Xy_tr_cs2", "Xm_te_cs"):
        if list(bundle[name].columns) != columns:
            found.append(f"{where}: {name} does not carry the declared features in order")
    if len(bundle["Xy_tr_cs2"]) != len(bundle["yy_trm"]):
        found.append(f"{where}: the YRBS training frame and its outcome are different lengths")
    if len(bundle["Xm_te_cs"]) != len(np.asarray(bundle["ymte"])):
        found.append(f"{where}: the MCS test frame and its outcome are different lengths")
    evaluable = np.asarray(bundle["ymte"], float)
    evaluable = evaluable[~np.isnan(evaluable)]
    if evaluable.size == 0:
        found.append(f"{where}: no MCS test respondent has a defined outcome")
    elif np.unique(evaluable).size < 2:
        found.append(f"{where}: the evaluable MCS test outcome has one class, so AUC is undefined")
    if np.unique(np.asarray(bundle["yy_trm"], float)).size < 2:
        found.append(f"{where}: the YRBS training outcome has one class, so nothing can be fitted")
    return found


problems = [problem for (seed, t), bundle in splits.items()
            for problem in check_reverse_frames(bundle, seed, t)]
if problems:
    raise ValueError(
        f"{len(problems)} canonical frame(s) do not align for the reverse reading; "
        f"first five: {problems[:5]}")

print(f"{len(splits)} canonical split bundles, all aligned for the reverse reading")
print("MCS test respondents with an undefined outcome stay in the frame and are dropped at "
      "metric time, exactly as in the forward analysis")

In [ ]:
# The YRBS fixed configurations, from the combined tracked specification. Nothing here selects
# one: the reverse-transfer model is trained on YRBS, so it takes the YRBS consensus selection,
# exactly as the forward `yrbs_local` reference does.
YRBS_SETTINGS = models.yrbs_settings()

expected = len(THRESHOLDS) * len(FAMILIES)
if len(YRBS_SETTINGS) != expected:
    raise ValueError(
        f"the YRBS specification carries {len(YRBS_SETTINGS)} cells, expected {expected} "
        f"({len(THRESHOLDS)} thresholds x {len(FAMILIES)} families)")
missing = [(t, f) for t in THRESHOLDS for f in FAMILIES if (t, f) not in YRBS_SETTINGS]
if missing:
    raise ValueError(f"the YRBS specification is missing {len(missing)} cell(s): {missing[:4]}")

print(f"{len(YRBS_SETTINGS)} YRBS configurations loaded from the tracked specification")
print("keyed (threshold, family) — no seed appears in the key, so every one of the twenty "
      "splits uses the same configuration")

In [ ]:
# The MCS local reference, reused from notebook 02 rather than refitted. The validation is the
# whole of the reuse: `mcs_local_reference_rows` refuses a frame from a different arm, lineage,
# configuration or grid, and refuses one short of a seed, rather than recomputing quietly.
battery = pd.read_csv(inputs.resolve("regime_battery.csv"))
local_reference_rows = transfer.mcs_local_reference_rows(
    battery, families=FAMILIES, thresholds=THRESHOLDS, seeds=SEEDS)

print(f"validated {len(local_reference_rows):,} source-reference rows from notebook 02 and "
      f"reused them as the MCS local reference")
print("no MCS model is refitted for that role, so this notebook fits one model per cell")

---
## C — Reverse models and the local reference

One call per cell fits the YRBS-configured role and returns its MCS predictions;
`transfer.reverse_transfer_scores` is the single definition of what it is, and
`transfer.reverse_metric_rows` turns the predictions into rows. The second role's rows were
validated and read in Section B and are concatenated here, so the finished frame carries both on
the same columns and nothing downstream needs to know which came from where.

**The predictions are MCS row-level and are never held.** Both roles score the MCS test frame,
so each score vector is restricted material. It is aggregated to a metric row where it is
produced and discarded there. No score frame is built, nothing is written, and no prediction is
printed.

**The metric definitions are the main pipeline's.** `evaluation.metrics` computes the AUC, the
PR-AUC, the Brier score, the ECE and the two calibration statistics for the forward battery and
computes them here. What differs is what leaves the function: the rates, and neither the counts
nor the denominators behind them.

In [ ]:
reverse_rows, _started = [], time.time()

for t in THRESHOLDS:
    for family in FAMILIES:
        for seed in SEEDS:
            # The configuration comes from the cohort the model is trained on. This one is
            # trained on YRBS, so it takes the YRBS consensus selection for the cell — the same
            # object at every seed, because the mapping carries no seed in its key.
            cell = transfer.reverse_transfer_scores(
                splits[(seed, t)], family=family, seed=seed,
                yrbs_params=YRBS_SETTINGS[(t, family)])
            if set(cell) != {"reverse_transfer"}:
                raise RuntimeError(
                    f"{(family, t, seed)} produced {sorted(cell)}, expected the one fitted "
                    f"role; a role missing here is invisible in the finished frame")
            # AGGREGATED IMMEDIATELY — the MCS score vectors in `cell` end their life on this
            # line and are never held, written or shown.
            reverse_rows += transfer.reverse_metric_rows(
                cell, family=family, threshold=t, seed=seed)
    print(f"  >={t} done  ({time.time() - _started:.0f}s elapsed)", flush=True)

# The one fitted role and the one reused role, on the same columns.
reverse_per_seed = pd.DataFrame(reverse_rows + local_reference_rows)

expected_rows = len(FAMILIES) * len(THRESHOLDS) * len(SEEDS) * len(ROLES)
if len(reverse_per_seed) != expected_rows:
    raise RuntimeError(
        f"the reverse frame holds {len(reverse_per_seed):,} rows for {expected_rows:,} "
        f"declared (family, threshold, seed, role) cells")
if set(reverse_per_seed["role"]) != set(ROLES):
    raise RuntimeError(
        f"the reverse frame carries roles {sorted(set(reverse_per_seed['role']))}, expected "
        f"{sorted(ROLES)}")

reverse_summary = evaluation.summarise_reverse_transfer(reverse_per_seed)
reverse_comparison = evaluation.reverse_transfer_comparison(reverse_summary)
print(f"\n{len(reverse_comparison):,} (family, threshold) cells summarised over "
      f"{len(SEEDS)} splits each")

In [ ]:
# How many seeds each role was estimated on. Three counts that sum to the seeds attempted, so a
# shortfall is visible here rather than inferred from a blank mean below.
estimability = (reverse_summary
                .groupby(["threshold", "role"])[["seeds_expected", "seeds_estimated",
                                                 "seeds_non_estimable"]].sum())
display(estimability)

---
## D — Results at the main threshold

`>=2` is the threshold the main text leads on. The two sensitivity thresholds follow in
Section E and are read on their own evidence.

**The mean and the spread are across twenty overlapping splits.** They describe performance and
its sensitivity to how the cohort was partitioned. **The standard deviation is not a confidence
interval** — the splits share most of their rows, so it is not a standard error and nothing
below reads it as one.

**The loss is a difference, and the mirror of the forward one.** `reverse_transfer_loss` is
`mcs_local_reference - reverse_transfer`, both evaluated on the same held-out MCS records: what
a model developed in YRBS gives up against a model developed in MCS. Forward, `transfer_loss` is
`mcs_internal - unadapted`. Same construction, cohorts exchanged.

**The quotient of two AUCs is deliberately not reported.** Neither is anchored at zero, so their
ratio is not a fraction of anything attainable. A chance-anchored share is not reported either:
that quantity was retired from the forward side because a model that adapts nothing still scores
highly on it, and it was read as a recovery when it was not.

In [ ]:
MAIN_COLUMNS = {
    "label": "family",
    "reverse_transfer_auc_mean": "reverse AUC",
    "reverse_transfer_auc_sd": "reverse SD",
    "mcs_local_reference_auc_mean": "MCS local AUC",
    "mcs_local_reference_auc_sd": "MCS local SD",
    "reverse_transfer_loss": "AUC loss",
    "reverse_transfer_prauc_mean": "reverse PR-AUC",
    "reverse_transfer_prauc_sd": "reverse PR SD",
    "mcs_local_reference_prauc_mean": "MCS local PR-AUC",
    "mcs_local_reference_prauc_sd": "MCS local PR SD",
    "reverse_transfer_prauc_loss": "PR-AUC loss",
}

main = reverse_comparison[reverse_comparison["threshold"] == HEADLINE_THRESHOLD]
if main.empty:
    raise ValueError(f"no reverse result at {HEADLINE_THRESHOLD}; the run did not cover it")

display(main[list(MAIN_COLUMNS)].rename(columns=MAIN_COLUMNS)
        .set_index("family").round(4))

blank = main[main["reverse_transfer_loss_reason"] != evaluation.ATTAINMENT_AVAILABLE]
if len(blank):
    print("the loss is blank where one of the two roles could not be estimated:")
    display(blank[["label", "reverse_transfer_loss_reason"]]
            .rename(columns={"label": "family", "reverse_transfer_loss_reason": "reason"})
            .set_index("family"))

In [ ]:
# Calibration at the same threshold, all three roles side by side. Discrimination and
# calibration measure different things and only one of them survives a monotone rescaling of
# the scores, so they are shown apart rather than folded into one table.
CALIBRATION_METRICS = ["cal_intercept_mean", "cal_intercept_sd", "cal_slope_mean",
                       "cal_slope_sd", "brier_mean", "brier_sd", "ece_mean", "ece_sd"]

calibration = reverse_summary[reverse_summary["threshold"] == HEADLINE_THRESHOLD].copy()
calibration["role"] = calibration["role"].map(regime_names.REVERSE_ROLE_DISPLAY)

# Reporting order comes from the family vocabulary, not from whatever order the groupby left.
family_order = {family: position for position, family in enumerate(FAMILIES)}
role_order = {regime_names.REVERSE_ROLE_DISPLAY[role]: position
              for position, role in enumerate(ROLES)}
calibration["_family"] = calibration["family"].map(family_order)
calibration["_role"] = calibration["role"].map(role_order)
calibration = calibration.sort_values(["_family", "_role"])

display(calibration.set_index(["label", "role"])[CALIBRATION_METRICS].round(4))

---
## E — Sensitivity at the other two thresholds

`>=1` and `>=3` are computed on the same splits under the same protocol. They are reported
compactly and read on their own evidence: whether they follow the pattern at `>=2` is a
question about these numbers, not an assumption the layout should make for a reader.

The prevalence ratio between the cohorts is not constant across the three cuts, so the three
rows of this table are not three measurements of one quantity.

In [ ]:
SENSITIVITY_COLUMNS = {
    "threshold": "threshold",
    "label": "family",
    "reverse_transfer_auc_mean": "reverse AUC",
    "mcs_local_reference_auc_mean": "MCS local AUC",
    "reverse_transfer_loss": "AUC loss",
    "reverse_transfer_prauc_loss": "PR-AUC loss",
}

sensitivity = reverse_comparison[
    reverse_comparison["threshold"].isin(SENSITIVITY_THRESHOLDS)]
missing = [t for t in SENSITIVITY_THRESHOLDS
           if t not in set(sensitivity["threshold"])]
if missing:
    raise ValueError(f"the run produced no reverse result at {missing}")

display(sensitivity[list(SENSITIVITY_COLUMNS)].rename(columns=SENSITIVITY_COLUMNS)
        .set_index(["threshold", "family"]).round(4))

blank = sensitivity[
    sensitivity["reverse_transfer_loss_reason"] != evaluation.ATTAINMENT_AVAILABLE]
if len(blank):
    print("the loss is blank at these cells, for the reason given:")
    display(blank[["threshold", "label", "reverse_transfer_loss_reason"]]
            .rename(columns={"label": "family",
                             "reverse_transfer_loss_reason": "reason"})
            .set_index(["threshold", "family"]))

---
## F — Aggregate output, and what this section does and does not establish

Two files, both aggregate, both under the internal exploration area outside the repository.

| file | grain |
|---|---|
| `reverse_transfer.csv` | one row per family x threshold x seed x role |
| `reverse_transfer_summary.csv` | one row per family x threshold, the three roles side by side |

The `mcs_local_reference` rows in both files are notebook 02's `mcs_internal` rows, validated
and relabelled by role. They are not a second computation of them.

**Candidate aggregate results; disclosure review is required before external sharing.** Every
row is derived from MCS records. No person-level prediction, respondent identifier, exact
count, denominator, fitted model, parameter file or intermediate score file is written, and
nothing here writes a publication candidate. Passing a structural check is not clearance.

In [ ]:
inputs.save_table(reverse_per_seed, "reverse_transfer.csv", subdir="exploration", quiet=True)
inputs.save_table(reverse_comparison, "reverse_transfer_summary.csv", subdir="exploration",
                  quiet=True)
print("\nCandidate aggregate results; disclosure review is required before external sharing.")

### What this section establishes, and what it cannot

**It measures one thing:** whether a model developed in YRBS and applied to MCS loses
discrimination against a model developed in MCS, family by family, at three thresholds, across
twenty splits.

**It cannot separate country, population and measurement.** One direction cannot, and neither
can two. What a second direction buys is a constraint on which explanations remain plausible,
and the reviewer response says so in those terms.

**The two directions face asymmetric label shift by construction**, so a difference between
them is not automatically a difference in transferability.

**The design is symmetric in its selection procedure.** Both cohorts' configurations were chosen
by the same three-seed consensus procedure inside their own outer-training partitions, and both
are fixed across the twenty splits, so the forward and reverse transfer losses are the same
construction with the cohorts exchanged. That alignment does not establish that any residual
model-development optimism is of the same magnitude in the two cohorts.

**Both local references are internal, not independent external validation.** Each is a model
developed and evaluated inside one cohort, on overlapping resamples.

**One of the two roles is not computed here.** `mcs_local_reference` is read from the forward
battery under validation. That makes it exactly the battery's source reference, which is the
point — but it also means this notebook does not independently confirm it, and a battery written
from a different code state would be caught by the checks only where the checks look.

**No adaptation is measured here at all.** Nothing below the unadapted rung was run in this
direction, so what an importing authority could buy back going YRBS to MCS is undetermined
rather than settled.